# Regresja liniowa w scikit-learn: trening i ocena modelu

**Plan notatnika:**
1. Dane
2. Podział na zbiór treningowy i testowy
3. Trening modelu
4. Predykcja
5. Ocena: zbiór treningowy vs testowy
6. Błędy predykcji
7. Metryki regresji: MAE, MSE, RMSE, R²
8. Punkt odniesienia (baseline)
9. Podsumowanie

## 1. Dane

Wygenerujemy sztuczne dane: jedna cecha i zmienna docelowa z szumem.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn
from sklearn.datasets import make_regression

sklearn.__version__

In [ ]:
data, target = make_regression(n_samples=300, n_features=1, noise=25.0, random_state=42)

print(f'data shape: {data.shape}')
print(f'target shape: {target.shape}')

In [ ]:
plt.figure(figsize=(8, 5))
plt.scatter(data, target)
plt.title('Dane')
plt.xlabel('cecha x')
plt.ylabel('zmienna docelowa')
plt.show()

## 2. Podział na zbiór treningowy i testowy

Model uczymy na **zbiorze treningowym**, a jakość oceniamy na **zbiorze testowym**, którego model nie widział (więcej w notatniku 2.3/06). Parametr `random_state` zapewnia powtarzalność podziału.

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(data, target, test_size=0.25, random_state=42)

print(f'X_train shape: {X_train.shape}')
print(f'X_test shape: {X_test.shape}')
print(f'y_train shape: {y_train.shape}')
print(f'y_test shape: {y_test.shape}')

In [ ]:
plt.figure(figsize=(8, 5))
plt.scatter(X_train, y_train, label='zbiór treningowy', color='gray', alpha=0.5)
plt.scatter(X_test, y_test, label='zbiór testowy', color='gold', alpha=0.7)
plt.title('Podział danych')
plt.xlabel('cecha x')
plt.ylabel('zmienna docelowa')
plt.legend()
plt.show()

## 3. Trening modelu

Model z jedną cechą ma postać:

$$\hat{y} = w_0 + w_1 x$$

Metoda `fit()` dopasowuje model do **danych treningowych**.

In [ ]:
from sklearn.linear_model import LinearRegression

regressor = LinearRegression()
regressor.fit(X_train, y_train)

print(f'w0 (intercept_) = {regressor.intercept_:.3f}')
print(f'w1 (coef_)      = {regressor.coef_[0]:.3f}')
print(f'Model: y = {regressor.intercept_:.3f} + {regressor.coef_[0]:.3f} * x')

In [ ]:
plt.figure(figsize=(8, 5))
plt.scatter(X_train, y_train, label='zbiór treningowy', color='gray', alpha=0.5)
plt.scatter(X_test, y_test, label='zbiór testowy', color='gold', alpha=0.7)
plt.plot(data, regressor.predict(data), color='red', linewidth=2, label='model')
plt.title('Dopasowany model')
plt.xlabel('cecha x')
plt.ylabel('zmienna docelowa')
plt.legend()
plt.show()

## 4. Predykcja

Metoda `predict()` zwraca przewidywane wartości. Dla jednej nowej wartości cechy też podajemy tablicę 2D (wiersze = obserwacje, kolumny = cechy).

In [ ]:
regressor.predict([[1.5]])

## 5. Ocena: zbiór treningowy vs testowy

Metoda `score()` zwraca dla regresji współczynnik $R^2$ (patrz sekcja 7).

In [ ]:
print(f'R2 na zbiorze treningowym: {regressor.score(X_train, y_train):.3f}')
print(f'R2 na zbiorze testowym:    {regressor.score(X_test, y_test):.3f}')

Wynik na zbiorze testowym jest zwykle trochę gorszy niż na treningowym. **Duża różnica** wskazywałaby na przeuczenie (rozdział 4.3). Decyduje wynik na zbiorze testowym.

## 6. Błędy predykcji

Zestawmy wartości rzeczywiste z przewidywanymi dla zbioru testowego i policzmy błąd każdej predykcji.

In [ ]:
y_true = y_test
y_pred = regressor.predict(X_test)

predictions = pd.DataFrame({'y_true': y_true, 'y_pred': y_pred})
predictions['error'] = predictions['y_true'] - predictions['y_pred']
predictions.head()

In [ ]:
predictions['error'].plot(kind='hist', bins=25, figsize=(8, 5), edgecolor='black')
plt.title('Rozkład błędów na zbiorze testowym')
plt.xlabel('błąd (y_true - y_pred)')
plt.show()

Pojedyncze wiersze bywają mylące (raz model trafia, raz nie). Jakość oceniamy **metrykami policzonymi na całym zbiorze testowym**.

## 7. Metryki regresji: MAE, MSE, RMSE, R²

![Metryki regresji](../obrazki/3.1_metryki_regresji.png)

| Metryka | Mówi | Jednostka |
|---|---|---|
| **MAE** – średni błąd bezwzględny | „średnio mylimy się o tyle” | taka jak $y$ |
| **MSE** – średni błąd kwadratowy | mocno karze duże błędy | kwadrat jednostki $y$ |
| **RMSE** – pierwiastek z MSE | jak MSE, ale w jednostce $y$ | taka jak $y$ |
| **R²** – współczynnik determinacji | jaka część zmienności $y$ jest wyjaśniona | bez jednostki |

Dla każdej metryki zaimplementujemy wzór ręcznie i porównamy z gotową funkcją z `sklearn.metrics`.

### MAE – Mean Absolute Error

$$MAE = \frac{1}{n}\sum_{i=1}^{n}|y_i - \hat{y}_i|$$

In [ ]:
from sklearn.metrics import mean_absolute_error

mae_custom = np.abs(y_true - y_pred).mean()

print(f'MAE (ręcznie): {mae_custom:.4f}')
print(f'MAE (sklearn): {mean_absolute_error(y_true, y_pred):.4f}')

### MSE – Mean Squared Error

$$MSE = \frac{1}{n}\sum_{i=1}^{n}(y_i - \hat{y}_i)^2$$

In [ ]:
from sklearn.metrics import mean_squared_error

mse_custom = ((y_true - y_pred) ** 2).mean()

print(f'MSE (ręcznie): {mse_custom:.4f}')
print(f'MSE (sklearn): {mean_squared_error(y_true, y_pred):.4f}')

### RMSE – Root Mean Squared Error

$$RMSE = \sqrt{MSE}$$

In [ ]:
from sklearn.metrics import root_mean_squared_error

rmse_custom = np.sqrt(mse_custom)

print(f'RMSE (ręcznie): {rmse_custom:.4f}')
print(f'RMSE (sklearn): {root_mean_squared_error(y_true, y_pred):.4f}')

### R² – współczynnik determinacji

$$R^2 = 1 - \frac{\sum_{i}(y_i - \hat{y}_i)^2}{\sum_{i}(y_i - \bar{y})^2}$$

Licznik to błąd modelu, mianownik – błąd „modelu”, który zawsze zgaduje średnią $\bar{y}$. Stąd interpretacja:
- **bliskie 1** – model dobrze wyjaśnia dane,
- **około 0** – model nie jest lepszy niż przewidywanie średniej,
- **poniżej 0** – model jest gorszy niż przewidywanie średniej.

In [ ]:
from sklearn.metrics import r2_score

r2_custom = 1 - ((y_true - y_pred) ** 2).sum() / ((y_true - y_true.mean()) ** 2).sum()

print(f'R2 (ręcznie): {r2_custom:.4f}')
print(f'R2 (sklearn): {r2_score(y_true, y_pred):.4f}')

### MAE czy RMSE?

RMSE jest zawsze nie mniejsze niż MAE, a **różnica rośnie, gdy są pojedyncze duże błędy**. Przykład: te same średnie błędy, ale w drugim przypadku jeden duży.

In [ ]:
ref = np.zeros(4)
rownomierne = np.array([5, 5, 5, 5])       # cztery błędy po 5
jeden_duzy = np.array([0, 0, 0, 20])       # jeden błąd 20

for nazwa, blad in [('równomierne błędy', rownomierne), ('jeden duży błąd', jeden_duzy)]:
    print(f'{nazwa}: MAE = {mean_absolute_error(ref, blad):.1f}, RMSE = {root_mean_squared_error(ref, blad):.1f}')

Oba zestawy mają to samo MAE, ale RMSE wyraźnie „karze” duży błąd. Wybór metryki zależy od problemu: gdy duże pomyłki są szczególnie kosztowne, patrzymy na RMSE.

## 8. Punkt odniesienia (baseline)

Czy wynik jest dobry? Zawsze porównuj model z prostym punktem odniesienia. `DummyRegressor` zawsze przewiduje **średnią** ze zbioru treningowego.

In [ ]:
from sklearn.dummy import DummyRegressor

dummy = DummyRegressor(strategy='mean')
dummy.fit(X_train, y_train)
y_dummy = dummy.predict(X_test)

wyniki = pd.DataFrame({
    'MAE': [mean_absolute_error(y_true, y_dummy), mean_absolute_error(y_true, y_pred)],
    'RMSE': [root_mean_squared_error(y_true, y_dummy), root_mean_squared_error(y_true, y_pred)],
    'R2': [r2_score(y_true, y_dummy), r2_score(y_true, y_pred)],
}, index=['baseline (średnia)', 'regresja liniowa'])
wyniki.round(3)

Model jest wyraźnie lepszy od przewidywania średniej: mniejsze błędy i wyższe $R^2$ (dla baseline $R^2 \approx 0$).

## 9. Podsumowanie

1. Model uczymy na zbiorze treningowym, a oceniamy na testowym.
2. Pojedyncze predykcje mogą mylić, liczą się metryki dla całego zbioru.
3. MAE i RMSE mówią o wielkości błędu (w jednostce $y$), a $R^2$ o tym, jak dobrze model wyjaśnia dane.
4. Warto raportować kilka metryk i porównywać model z baseline.
5. Duża różnica między wynikiem na treningu i teście sygnalizuje przeuczenie.